In [56]:
# CELLULE 1 — Installation des dépendances
!pip install datasets


[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [57]:
# CELLULE 2 — Chargement du dataset MedQuAD
from datasets import load_dataset

dataset = load_dataset("lavita/MedQuAD")

In [58]:
# CELLULE 3 — Exploration du dataset
print("Colonnes :", dataset["train"].column_names)

import pprint
print("\nExemple brut :")
pprint.pprint(dataset["train"][0])

Colonnes : ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer']

Exemple brut :
{'answer': 'Keratoderma with woolly hair is a group of related conditions that '
           'affect the skin and hair and in many cases increase the risk of '
           'potentially life-threatening heart problems. People with these '
           'conditions have hair that is unusually coarse, dry, fine, and '
           'tightly curled. In some cases, the hair is also sparse. The woolly '
           'hair texture typically affects only scalp hair and is present from '
           'birth. Starting early in life, affected individuals also develop '
           'palmoplantar keratoderma, a condition that causes skin on the '
           'palms of the hands and the soles of the feet to become thick, '
           'scaly, and calloused.  Cardiomyopathy, which is a d

In [59]:
# CELLULE 4 — Transformation (on garde uniquement la réponse)
def transform(example):
    return {
        "response": example.get("answer", ""),
        "source": "medquad"
    }

In [60]:
# CELLULE 5 — Application de la transformation
clean_dataset = dataset["train"].map(transform)

In [61]:
# CELLULE 6 — Filtrage des lignes vides
clean_dataset = clean_dataset.filter(
    lambda x: x["response"] != ""
)

In [62]:
# CELLULE 7 — Suppression des colonnes inutiles
clean_dataset = clean_dataset.remove_columns(
    [col for col in clean_dataset.column_names if col not in ["response", "source"]]
)

In [63]:
# CELLULE 8 — Vérification
print("Nombre d'exemples :", len(clean_dataset))
print("\nExemple nettoyé :")
print(clean_dataset[0])

Nombre d'exemples : 47441

Exemple nettoyé :
{'response': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appear until adolescence or later. Complications of cardiomyopathy can include an abnormal hea

In [64]:
# CELLULE 9 — Sauvegarde
clean_dataset.save_to_disk("data/processed/medquad_clean")

Saving the dataset (1/1 shards): 100%|██████████| 47441/47441 [00:00<00:00, 232475.95 examples/s]


In [65]:
# CELLULE 10 — Règles de triage médical
symptom_data = {
    "douleur thoracique": {
        "response": "Cela peut être un signe d'urgence cardiaque. Appelez immédiatement les urgences.",
        "triage": "urgent"
    },
    "essoufflement": {
        "response": "Cela peut indiquer un problème respiratoire grave. Consultez immédiatement.",
        "triage": "urgent"
    },
    "fièvre": {
        "response": "Cela peut être une infection. Surveillez et consultez si cela persiste.",
        "triage": "modéré"
    },
    "vomissements": {
        "response": "Hydratez-vous et consultez si cela continue.",
        "triage": "modéré"
    },
    "maux de tête": {
        "response": "Cela peut être bénin. Reposez-vous et surveillez.",
        "triage": "faible"
    },
    "fatigue": {
        "response": "Cela peut être lié au manque de repos. Surveillez votre état.",
        "triage": "faible"
    }
}

In [66]:
# CELLULE 11 — Génération des cas patients
import random

def create_triage(example):
    symptom = random.choice(list(symptom_data.keys()))
    
    return {
        "instruction": f"J’ai {symptom} depuis quelques heures, que faire ?",
        "response": symptom_data[symptom]["response"],
        "triage_level": symptom_data[symptom]["triage"],
        "source": "synthetic"
    }

In [67]:
# CELLULE 12 — Création du dataset de triage
triage_dataset = clean_dataset.map(create_triage)

In [68]:
# CELLULE 13 — Vérification
print("Exemple triage :")
print(triage_dataset[0])

Exemple triage :
{'response': 'Cela peut être une infection. Surveillez et consultez si cela persiste.', 'source': 'synthetic', 'instruction': 'J’ai fièvre depuis quelques heures, que faire ?', 'triage_level': 'modéré'}


In [69]:
# CELLULE 14 — Ajout des métadonnées
def enrich_metadata(example):
    return {
        "instruction": example.get("instruction", ""),
        "response": example.get("response", ""),
        "triage_level": example.get("triage_level", ""),
        "symptoms": example.get("instruction", ""),
        "language": "fr",
        "confidence_score": 0.9,
        "source": example.get("source", "synthetic")
    }

In [70]:
# CELLULE 15 — Application enrichissement
triage_dataset = triage_dataset.map(enrich_metadata)

Map: 100%|██████████| 47441/47441 [00:04<00:00, 9552.68 examples/s] 


In [71]:
# CELLULE 16 — Nettoyage final des colonnes
triage_dataset = triage_dataset.remove_columns(
    [col for col in triage_dataset.column_names if col not in [
        "instruction", "response", "triage_level",
        "symptoms", "language", "confidence_score", "source"
    ]]
)

In [72]:
# CELLULE 17 — Vérification finale
print("Colonnes finales :", triage_dataset.column_names)
print("\nExemple final :")
print(triage_dataset[0])

Colonnes finales : ['response', 'source', 'instruction', 'triage_level', 'symptoms', 'language', 'confidence_score']

Exemple final :
{'response': 'Cela peut être une infection. Surveillez et consultez si cela persiste.', 'source': 'synthetic', 'instruction': 'J’ai fièvre depuis quelques heures, que faire ?', 'triage_level': 'modéré', 'symptoms': 'J’ai fièvre depuis quelques heures, que faire ?', 'language': 'fr', 'confidence_score': 0.9}


In [73]:
# CELLULE 18 — Sauvegarde finale
triage_dataset.save_to_disk("data/final/triage_dataset_v1")

Saving the dataset (1/1 shards): 100%|██████████| 47441/47441 [00:00<00:00, 799509.71 examples/s]


In [74]:
# CELLULE 19 — Réorganisation des colonnes (ordre propre)

triage_dataset = triage_dataset.select_columns([
    "instruction",
    "response",
    "triage_level",
    "symptoms",
    "language",
    "confidence_score",
    "source"
])

In [75]:
# CELLULE 20 — Vérification finale

print("Colonnes finales propres :", triage_dataset.column_names)
print("\nExemple final propre :")
print(triage_dataset[0])

Colonnes finales propres : ['instruction', 'response', 'triage_level', 'symptoms', 'language', 'confidence_score', 'source']

Exemple final propre :
{'instruction': 'J’ai fièvre depuis quelques heures, que faire ?', 'response': 'Cela peut être une infection. Surveillez et consultez si cela persiste.', 'triage_level': 'modéré', 'symptoms': 'J’ai fièvre depuis quelques heures, que faire ?', 'language': 'fr', 'confidence_score': 0.9, 'source': 'synthetic'}


In [76]:
# CELLULE 21 — Sauvegarde finale propre

triage_dataset.save_to_disk("data/final/triage_dataset_v2")

Saving the dataset (1/1 shards): 100%|██████████| 47441/47441 [00:00<00:00, 388487.95 examples/s]


In [77]:
# CELLULE 22 — Split train / validation / test

split_dataset = triage_dataset.train_test_split(test_size=0.3, seed=42)

train_val = split_dataset["train"].train_test_split(test_size=0.1765, seed=42)  
# ≈ 70% train, 15% val, 15% test

dataset_splits = {
    "train": train_val["train"],
    "validation": train_val["test"],
    "test": split_dataset["test"]
}

print("Train:", len(dataset_splits["train"]))
print("Validation:", len(dataset_splits["validation"]))
print("Test:", len(dataset_splits["test"]))

Train: 27346
Validation: 5862
Test: 14233


In [78]:
# CELLULE 23 — Sauvegarde des splits

dataset_splits["train"].save_to_disk("data/final/train")
dataset_splits["validation"].save_to_disk("data/final/validation")
dataset_splits["test"].save_to_disk("data/final/test")

Saving the dataset (1/1 shards): 100%|██████████| 14233/14233 [00:00<00:00, 115579.77 examples/s]


In [79]:
# CELLULE 24 — Distribution des classes

from collections import Counter

print("Distribution train :", Counter(dataset_splits["train"]["triage_level"]))

Distribution train : Counter({'urgent': 9320, 'faible': 9030, 'modéré': 8996})


In [80]:
# CELLULE 25 — Création dataset SFT (5000 exemples)

sft_dataset = dataset_splits["train"].shuffle(seed=42).select(range(5000))

print("Taille SFT :", len(sft_dataset))
print(sft_dataset[0])

Taille SFT : 5000
{'instruction': 'J’ai vomissements depuis quelques heures, que faire ?', 'response': 'Hydratez-vous et consultez si cela continue.', 'triage_level': 'modéré', 'symptoms': 'J’ai vomissements depuis quelques heures, que faire ?', 'language': 'fr', 'confidence_score': 0.9, 'source': 'synthetic'}


In [81]:
# CELLULE 26 — Sauvegarde SFT

sft_dataset.save_to_disk("data/final/sft_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 5000/5000 [00:00<00:00, 31646.29 examples/s]


In [82]:
# CELLULE 27 — Création dataset DPO

def create_dpo(example):
    return {
        "instruction": example["instruction"],
        "chosen": example["response"],  # bonne réponse
        "rejected": "Je ne sais pas. Consultez un médecin si besoin.",  # mauvaise réponse volontaire
    }

dpo_dataset = sft_dataset.map(create_dpo)

print(dpo_dataset[0])

Map: 100%|██████████| 5000/5000 [00:00<00:00, 9481.52 examples/s]

{'instruction': 'J’ai vomissements depuis quelques heures, que faire ?', 'response': 'Hydratez-vous et consultez si cela continue.', 'triage_level': 'modéré', 'symptoms': 'J’ai vomissements depuis quelques heures, que faire ?', 'language': 'fr', 'confidence_score': 0.9, 'source': 'synthetic', 'chosen': 'Hydratez-vous et consultez si cela continue.', 'rejected': 'Je ne sais pas. Consultez un médecin si besoin.'}


In [83]:
# CELLULE 28 — Sauvegarde DPO

dpo_dataset.save_to_disk("data/final/dpo_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 5000/5000 [00:00<00:00, 189840.77 examples/s]


# Schéma des métadonnées

{
  "instruction": "Texte du patient décrivant ses symptômes",
  "response": "Réponse médicale adaptée",
  "triage_level": "Niveau d'urgence (urgent, modéré, faible)",
  "symptoms": "Résumé des symptômes",
  "language": "Langue du texte (fr/en)",
  "confidence_score": "Score de confiance (0 à 1)",
  "source": "Origine des données (medquad, synthetic)"
}

# RGPD — Justification


Le dataset utilisé dans ce projet a été construit à partir de sources publiques (MedQuAD) et de données synthétiques générées.

Aucune donnée personnelle identifiable (PII) n’est présente dans les données utilisées. Les textes ne contiennent ni noms, ni adresses, ni informations permettant d’identifier un individu.
Dans un contexte réel, un processus d’anonymisation basé sur des outils comme Presidio aurait été utilisé pour détecter et masquer automatiquement les données sensibles (noms, numéros, etc.).
Ainsi, le dataset respecte les principes du RGPD :


- minimisation des données
- anonymisation
- absence de données personnelles


